In [0]:
# Catalog name                       
catalog = 'workspace'

# CDC column                        #  column used to track and manage changes to data in a database
cdc_column = 'modifiedDate'
               
# Back-dated refresh
backdated_refresh = ""

# Source object                     
source_object = 'silver_bookings'

# Source schema
source_schema = 'silver'

# Target object                     
target_object = 'FactBookings'

# Target schema
target_schema = 'gold'

# Source fact table 
fact_table = f'{catalog}.{source_schema}.{source_object}'

# Fact key columns list (booking date was added as 2 records had same info just different booking date)
fact_key_cols = ['DimPassengersKey', 'DimFlightsKey', 'DimAirportsKey', 'booking_date']


In [0]:
dimensions = [
    {
        "table" : f'{catalog}.{target_schema}.dimpassengers',
        "alias" : 'dimpassengers',
        "join_keys" : [('passenger_id', 'passenger_id')]     # (fact_col, dim_col)
    },

    {
        "table" : f'{catalog}.{target_schema}.dimflights',
        "alias" : 'dimflights',
        "join_keys" : [('flight_id', 'flight_id')]     # (fact_col, dim_col)
    },

    {
        "table" : f'{catalog}.{target_schema}.dimairports',
        "alias" : 'dimairports',
        "join_keys" : [('airport_id', 'airport_id')]     # (fact_col, dim_col)
    }
]

# Columns you want to keep from fact table (numeric & date columns)
fact_columns = ['amount', 'booking_date', 'modifiedDate']

#### **Last load date**

In [0]:
if backdated_refresh == "":

  # If target table exists in destination
  if spark.catalog.tableExists(f'{catalog}.{target_schema}.{target_object}'):

    # Get the maximum modifiedDate (latest change) from the target table
    last_load = spark.sql(f'select max({cdc_column}) from {catalog}.{target_schema}.{target_object}').collect()[0][0]
  
  # If the target table does not exist
  else : 
    
    last_load = '1900-01-01 00:00:00'

# If backdated refresh is provided
else:
  # Use the given backdated refresh timestamp as last_load
  last_load = backdated_refresh

last_load  

datetime.datetime(2025, 9, 5, 13, 43, 57, 332000)

#### **Dynamic Fact query [Bring keys]**

In [0]:
def generate_fact_query_incremental (fact_table, dimensions, fact_columns, cdc_columns, processing_date):
    fact_alias = "f"

    # Base columns to select
    select_cols = [f'{fact_alias}.{col}' for col in fact_columns]

    # build joins dynamically
    join_clauses = []
    for dim in dimensions:
        table_full = dim['table']
        alias = dim['alias']
        table_name = table_full.split('.')[-1]
        surrogate_key = f'{alias}.{table_name}Key'
        select_cols.append(surrogate_key)

        # Build On clause
        on_condition = [f'{fact_alias}.{fk} = {alias}.{dk}' for fk, dk in dim['join_keys']]

        join_clause = f"LEFT JOIN {table_full} {alias} ON " + " AND ".join(on_condition)
        join_clauses.append(join_clause) 


    # Final select and join clause
    select_clause = ",\n       ".join(select_cols)
    joins = "\n".join(join_clauses)

    # Where clause for incremental filtering
    where_clause = f"{fact_alias}.{cdc_columns} >= DATE('{last_load}')"

    # Final query
    query = f"""
    SELECT
        {select_clause}
    FROM {fact_table} {fact_alias} {joins}
    WHERE 
        {where_clause}
        """.strip()

    return query   

In [0]:
query = generate_fact_query_incremental(fact_table, dimensions, fact_columns, cdc_column, last_load)
print(query)

SELECT
        f.amount,
       f.booking_date,
       f.modifiedDate,
       dimpassengers.dimpassengersKey,
       dimflights.dimflightsKey,
       dimairports.dimairportsKey
    FROM workspace.silver.silver_bookings f LEFT JOIN workspace.gold.dimpassengers dimpassengers ON f.passenger_id = dimpassengers.passenger_id
LEFT JOIN workspace.gold.dimflights dimflights ON f.flight_id = dimflights.flight_id
LEFT JOIN workspace.gold.dimairports dimairports ON f.airport_id = dimairports.airport_id
    WHERE 
        f.modifiedDate >= DATE('2025-09-05 13:43:57.332000')


#### **DF_Fact**

In [0]:
df_fact = spark.sql(query)

In [0]:
df_fact.display()

amount,booking_date,modifiedDate,dimpassengersKey,dimflightsKey,dimairportsKey
912.64,2025-07-10,2025-09-05T13:43:57.332Z,207,47,4
597.04,2025-07-01,2025-09-05T13:43:57.332Z,124,88,51
724.88,2025-07-10,2025-09-05T13:43:57.332Z,103,96,36
1370.99,2025-07-12,2025-09-05T13:43:57.332Z,49,69,1
1476.15,2025-07-01,2025-09-05T13:43:57.332Z,210,84,53
800.39,2025-06-26,2025-09-05T13:43:57.332Z,126,84,42
1187.22,2025-06-28,2025-09-05T13:43:57.332Z,77,13,55
913.43,2025-07-18,2025-09-05T13:43:57.332Z,118,5,33
260.75,2025-07-08,2025-09-05T13:43:57.332Z,90,102,34
245.61,2025-07-21,2025-09-05T13:43:57.332Z,12,80,45


#### **Upsert**

In [0]:
# Fact key columns merge condition
fact_key_cols_str = " AND ".join([f"src.{col} = trg.{col}" for col in fact_key_cols])
fact_key_cols_str

'src.DimPassengersKey = trg.DimPassengersKey AND src.DimFlightsKey = trg.DimFlightsKey AND src.DimAirportsKey = trg.DimAirportsKey AND src.booking_date = trg.booking_date'

In [0]:
from delta.tables import DeltaTable

if spark.catalog.tableExists(f"{catalog}.{target_schema}.{target_object}"):
    
    # Get a reference to the existing Delta table
    dlt_obj = DeltaTable.forName(spark, f"{catalog}.{target_schema}.{target_object}")

    # Start a merge (upsert) between source dataframe and target Delta table using surrogate key
    ( dlt_obj.alias("trg").merge(df_fact.alias("src"), fact_key_cols_str)\
            # Update if a match is found and only if source record is newer (based on CDC column)
            .whenMatchedUpdateAll(condition = f"src.{cdc_column} >= trg.{cdc_column}")\
            # Insert the row if it doesn’t exist in the target    
            .whenNotMatchedInsertAll()\
            .execute() )

 # If the target Delta table doesn’t exist
else:

    ( df_fact.write.format("delta")\
        # Append mode (creates the table if it doesn’t exist)
        .mode("append")\
        # Save the dataframe as a new managed Delta table as gold layer in the target schema
        .saveAsTable(f"{catalog}.{target_schema}.{target_object}") )               

In [0]:
%sql
select * from workspace.gold.factbookings

amount,booking_date,modifiedDate,dimpassengersKey,dimflightsKey,dimairportsKey
1368.6,2025-06-07,2025-09-05T13:43:57.332Z,177,56,29
427.56,2025-03-28,2025-09-05T13:43:57.332Z,177,51,3
320.52,2025-05-05,2025-09-05T13:43:57.332Z,177,41,9
220.26,2025-06-16,2025-09-05T13:43:57.332Z,177,94,39
293.92,2025-07-18,2025-09-05T13:43:57.332Z,63,10,9
687.89,2025-07-11,2025-09-05T13:43:57.332Z,63,11,26
716.33,2025-06-01,2025-09-05T13:43:57.332Z,63,39,1
940.65,2025-04-19,2025-09-05T13:43:57.332Z,63,63,23
265.87,2025-05-17,2025-09-05T13:43:57.332Z,63,43,45
835.11,2025-06-10,2025-09-05T13:43:57.332Z,63,97,42


No duplicate records should be present in any dimension table. To verify we can run below code for each dimension

In [0]:
from pyspark.sql.functions import *
df= spark.sql(f"select * from {catalog}.{target_schema}.dimpassengers").groupBy('dimpassengersKey').count().filter(col('count') > 1)
df.display()

dimpassengersKey,count


### **DBT**

dbt (data build tool) is primarily used for the transformation step in ETL/ELT pipelines. It follows a SQL-first approach, which makes writing and debugging transformations easier compared to code-heavy frameworks like PySpark. In this project, we will use dbt to create curated business views that will be consumed by stakeholders which were saved in dbt_sahilsurve schema.

In [0]:
%sql

-- Countries and their respective total amount sorted in descending order
select * from workspace.dbt_sahilsurve.countries_and_amount

Country,Total_Amount
South Georgia and the South Sandwich Islands,55720.82
Libyan Arab Jamahiriya,37338.86
Switzerland,37338.81
Cote d'Ivoire,28785.21
Cayman Islands,28575.56
Macedonia,26128.29
Korea,25090.83
Tokelau,25013.71
Hong Kong,24416.46
Ireland,24156.39
